# Clustering: Cardiovascular laboratory
Clustering to distinguish patients by metabolic and electrolyte instability

Before clustering, we import the dependency libraries: 

In [ ]:
!pip install pandas scikit-learn

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from collections import Counter
from pathlib import Path

And load the patient profile dataset 

In [ ]:
notebook_dir = Path().resolve()
FEATURE_PATH = notebook_dir.parents[1]  / "1" / "Features" 
PATIENT_PROFILES_NAME = "patient_profiles.csv"

df = pd.read_csv(FEATURE_PATH / PATIENT_PROFILES_NAME)

## Feature selection for  cardiovascular severity clustering

We take from patient profiles features all those which rely on cardiovascular severity

In [ ]:
features_laboratory = 

In [ ]:
# Extract data subset
df_severity = df[['subject_id'] + features_laboratory].copy()

In [ ]:
# Handle missing values
df_severity_clean = df_severity.dropna(subset=features_laboratory)
print(f"\nUsable patients: {len(df_severity_clean)} / {len(df_severity)}")


Usable patients: 4694 / 4694


In [ ]:
# Prepare data for clustering
X_severity = df_severity_clean[features_laboratory].values
subject_ids_severity = df_severity_clean['subject_id'].values


## Data normalization

We use `StandardScaler` from sklearn to normalize data (Z-score normalization)

In [ ]:
scaler = StandardScaler()
X_severity_normalized = scaler.fit_transform(X_severity)

## K-means clustering 

### Optimal K derivation

In [ ]:
# Range of k to test
k_range = range(2, 11)

# Evaluation metrics
inertias = []
silhouette_scores = []
calinski_scores = []
davies_bouldin_scores = []

print("\nComputing metrics for k from 2 to 10...")
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels = kmeans.fit_predict(X_severity_normalized)

    inertia = kmeans.inertia_
    silhouette = silhouette_score(X_severity_normalized, labels)

    inertias.append(inertia)
    silhouette_scores.append(silhouette)
    
    print(f"k={k:2d} | Inertia: {inertia:8.2f} | Silhouette: {silhouette:.4f}")


Computing metrics for k from 2 to 10...
k= 2 | Inertia: 67391.89 | Silhouette: 0.1925
k= 3 | Inertia: 62680.64 | Silhouette: 0.1921
k= 4 | Inertia: 58471.90 | Silhouette: 0.1424
k= 5 | Inertia: 54922.30 | Silhouette: 0.1371
k= 6 | Inertia: 51097.46 | Silhouette: 0.1393
k= 7 | Inertia: 47813.29 | Silhouette: 0.1562
k= 8 | Inertia: 43606.19 | Silhouette: 0.1857
k= 9 | Inertia: 41898.12 | Silhouette: 0.1493
k=10 | Inertia: 38396.00 | Silhouette: 0.1855


In [ ]:
best_silhouette_k = list(k_range)[np.argmax(silhouette_scores)]
print(f" Optimal Silhouette Score: k = {best_silhouette_k}")

 Optimal Silhouette Score: k = 2


In [ ]:
optimal_k = best_silhouette_k

### Training with optimal K

In [ ]:
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10, max_iter=300)
cluster_labels = kmeans_final.fit_predict(X_severity_normalized)

In [ ]:
# Add labels to dataframe
df_severity_clean['cluster'] = cluster_labels

### Clusters distribution and characterization

In [ ]:
cluster_counts = pd.Series(cluster_labels).value_counts().sort_index()
print("\nCluster sizes:")
for cluster_id, count in cluster_counts.items():
    percentage = (count / len(cluster_labels)) * 100
    print(f"  Cluster {cluster_id}: {count:4d} patients ({percentage:5.1f}%)")


Cluster sizes:
  Cluster 0: 1448 patients ( 30.8%)
  Cluster 1: 3246 patients ( 69.2%)


#### Centroids

In [ ]:
# Centroids in normalized space
centroids_normalized = kmeans_final.cluster_centers_

# Inverse transform for interpretability
centroids_original = scaler.inverse_transform(centroids_normalized)

# Create dataframe with centroids
centroids_df = pd.DataFrame(
    centroids_original,
    columns=features_laboratory,
    index=[f'Cluster {i}' for i in range(optimal_k)]
)

print("\nCentroids (original values):")
print(centroids_df.round(3))


Centroids (original values):
           heart_failure_severity_score  cad_severity_score  lv_dilated  \
Cluster 0                         2.699               0.888       0.785   
Cluster 1                         0.814               0.355       0.625   

           rv_dysfunction  wall_motion_abnormality  mitral_regurgitation  \
Cluster 0           0.577                    0.920                 0.883   
Cluster 1           0.009                    0.188                 0.710   

           atrial_fibrillation   lbbb  q_waves  st_depression  st_elevation  \
Cluster 0                0.120  0.056    0.131          0.116         0.068   
Cluster 1                0.132  0.033    0.042          0.062         0.022   

           oxygen_saturation  creat_max  creat_abnormal_ratio  \
Cluster 0             97.586      2.221                 0.503   
Cluster 1             97.080      1.564                 0.338   

           total_procedures  procedures_days_span  
Cluster 0             4.842  